# Machine Learning RUL Prediction: Random Forest vs XGBoost

Thực nghiệm huấn luyện và đánh giá hai mô hình:
1. **Random Forest Regressor** (Baseline)
2. **XGBoost Regressor** (Mô hình chính)

Các chỉ số đánh giá:
- MAE (Mean Absolute Error)
- RMSE (Root Mean Squared Error)
- R² Score
- NASA PHM Asymmetric Score
- Thời gian huấn luyện & độ trễ suy luận (inference latency)

In [ ]:
from src.preprocessing.loader import load_data
from src.preprocessing.label_rul import add_rul_target
from src.preprocessing.clean import prune_low_variance_sensors, clean_dataframe
from src.preprocessing.feature_engineering import extract_rolling_features, get_feature_columns
from src.training.train_rf import train_random_forest_regressor
from src.training.train_xgboost import train_xgboost_regressor
from src.training.evaluate import evaluate_model, generate_comparison_table

# 1. Load data & compute rolling features
train_df = load_data(mode="train")
test_df = load_data(mode="test")

train_df = add_rul_target(train_df)
test_df = add_rul_target(test_df)

train_clean, dropped = prune_low_variance_sensors(clean_dataframe(train_df))
test_clean, _ = prune_low_variance_sensors(clean_dataframe(test_df), explicit_drop=dropped)

train_feat = extract_rolling_features(train_clean)
test_feat = extract_rolling_features(test_clean)

features = get_feature_columns(train_feat)
print(f"Engineered {len(features)} total features.")

X_train, y_train = train_feat[features], train_feat["RUL"]
X_test, y_test = test_feat[features], test_feat["RUL"]

In [ ]:
# 2. Train and evaluate Random Forest
rf_model, rf_info = train_random_forest_regressor(X_train, y_train)
rf_eval = evaluate_model(rf_model, X_test, y_test, model_name="Random Forest")

In [ ]:
# 3. Train and evaluate XGBoost
xgb_model, xgb_info = train_xgboost_regressor(X_train, y_train)
xgb_eval = evaluate_model(xgb_model, X_test, y_test, model_name="XGBoost")

In [ ]:
# 4. Display Comparison Table
from IPython.display import Markdown
table_md = generate_comparison_table([rf_eval, xgb_eval])
Markdown(table_md)